# Real Estate Scraper Maintenance Guide

This notebook provides guidance on maintaining and improving the real estate scraping codebase, with a focus on:

1. **Handling Deprecated Files**: How to identify, mark, and handle deprecated components
2. **Improving Error Handling**: Implementing robust error handling and logging mechanisms
3. **Updating Documentation**: Ensuring documentation stays up-to-date with codebase changes

This guide is intended for developers maintaining the real estate scraping tools in the `immob/api_immobiliare` package.

Author: Lucas P  
Date: July 6, 2025

## 1. Setup and Imports

First, let's import the necessary libraries and set up our environment. We'll need:

- `os` and `pathlib` for file operations
- `importlib` for dynamic module imports
- `warnings` for warning management
- `logging` for improved logging
- `inspect` for introspection of modules

In [ ]:
import os
import sys
import re
import importlib
import inspect
import warnings
import logging
import functools
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set, Callable

# Configure better logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add the parent directory to sys.path so we can import our modules
module_path = str(Path.cwd().parent)
if module_path not in sys.path:
    sys.path.append(module_path)

print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")

# Create a nice banner function to separate sections
def print_section(title):
    """Print a section title with decorative formatting"""
    terminal_width = 80
    print(f"\n{'=' * terminal_width}")
    print(f"{title.center(terminal_width)}")
    print(f"{'=' * terminal_width}\n")

## 2. Identifying and Handling Deprecated Files

As our codebase evolves, some files become obsolete or are replaced by newer implementations. It's important to:

1. Identify deprecated files
2. Mark them with appropriate warnings
3. Create migration paths for users
4. Eventually remove them when appropriate

In this section, we'll demonstrate techniques for managing deprecated code in our real estate scraping project.

In [ ]:
# Function to scan a directory for Python files and analyze them
def scan_for_potential_deprecated_files(directory: Path, reference_files: List[str]) -> Dict[str, Dict]:
    """
    Scan a directory for Python files and identify potential deprecated ones.
    
    Args:
        directory: Directory to scan
        reference_files: List of known current files that should not be deprecated
        
    Returns:
        Dictionary of potential deprecated files with analysis
    """
    print_section("SCANNING FOR POTENTIALLY DEPRECATED FILES")
    
    # Make sure directory exists
    if not directory.exists() or not directory.is_dir():
        logger.error(f"Directory {directory} does not exist")
        return {}
    
    # Get all Python files
    python_files = list(directory.glob("**/*.py"))
    logger.info(f"Found {len(python_files)} Python files in {directory}")
    
    # Convert reference files to absolute paths for comparison
    reference_paths = [directory / ref for ref in reference_files]
    
    # Analysis results
    results = {}
    
    for file_path in python_files:
        # Skip __init__.py files and reference files
        if file_path.name == "__init__.py" or file_path in reference_paths:
            continue
            
        rel_path = file_path.relative_to(directory)
        
        # Check for deprecated markers in the file
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
            # Look for deprecation markers
            is_marked_deprecated = any(marker in content.lower() for marker in 
                                     ['deprecated', 'obsolete', 'do not use'])
            
            # Look for imports to see what this file depends on
            import_matches = re.findall(r'^from\s+(\S+)\s+import|^import\s+(\S+)', 
                                        content, re.MULTILINE)
            imports = []
            for match in import_matches:
                imports.extend([m for m in match if m])
                
            # Look for what might be importing this file
            module_name = str(rel_path).replace('/', '.').replace('\\', '.').replace('.py', '')
            imported_by = []
            for other_path in python_files:
                if other_path == file_path:
                    continue
                    
                with open(other_path, 'r', encoding='utf-8', errors='ignore') as of:
                    other_content = of.read()
                    if module_name in other_content:
                        imported_by.append(str(other_path.relative_to(directory)))
            
            # Look for modification time
            mod_time = datetime.fromtimestamp(file_path.stat().st_mtime)
            
            # Add to results
            results[str(rel_path)] = {
                'path': str(file_path),
                'is_marked_deprecated': is_marked_deprecated,
                'imports': imports,
                'imported_by': imported_by,
                'last_modified': mod_time,
                'size': file_path.stat().st_size
            }
    
    # Sort results by whether they're imported and last modified date
    sorted_results = {}
    for key, value in sorted(results.items(), 
                            key=lambda x: (len(x[1]['imported_by']), x[1]['last_modified']),
                            reverse=True):
        sorted_results[key] = value
    
    return sorted_results

# Define the directory to scan and reference files that are known to be current
directory = Path(module_path)
reference_files = [
    'data_manager.py',
    'retrievers.py', 
    'advanced_features.py', 
    'model_utils.py', 
    'visualizations.py'
]

# Scan for potential deprecated files
deprecated_file_analysis = scan_for_potential_deprecated_files(directory, reference_files)

# Display results
if deprecated_file_analysis:
    print(f"Found {len(deprecated_file_analysis)} potential deprecated files:")
    
    for file, info in deprecated_file_analysis.items():
        status = "MARKED DEPRECATED" if info['is_marked_deprecated'] else "POTENTIAL DEPRECATION"
        imported_count = len(info['imported_by'])
        
        print(f"\n{file} - {status}")
        print(f"  Last modified: {info['last_modified']}")
        print(f"  Imported by {imported_count} files")
        
        if imported_count > 0:
            print(f"  Imported by: {', '.join(info['imported_by'])}")
else:
    print("No potential deprecated files found")

In [ ]:
# Function to add a deprecation notice to a file
def mark_file_as_deprecated(
    file_path: Path, 
    alternative: str, 
    removal_date: Optional[str] = None
) -> bool:
    """
    Add a deprecation notice to a Python file.
    
    Args:
        file_path: Path to the file to mark
        alternative: Alternative module/function to use
        removal_date: When this file will be removed (optional)
        
    Returns:
        True if successful, False otherwise
    """
    if not file_path.exists():
        logger.error(f"File {file_path} does not exist")
        return False
    
    try:
        # Read the file
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # Check if it's already marked
        if "DEPRECATED" in content:
            logger.info(f"File {file_path} is already marked as deprecated")
            return True
        
        # Create the deprecation notice
        removal_notice = f" It will be removed on {removal_date}." if removal_date else ""
        deprecation_notice = f'''"""
DEPRECATED: This file is deprecated and should not be used.
Please use {alternative} instead.{removal_notice}
"""

import warnings
warnings.warn(
    f"The module {file_path.stem} is deprecated. "
    f"Please use {alternative} instead.{removal_notice}",
    DeprecationWarning,
    stacklevel=2
)

'''
        
        # Add the notice to the beginning of the file
        new_content = deprecation_notice + content
        
        # Create a backup
        backup_path = file_path.with_suffix(f".py.bak")
        shutil.copy2(file_path, backup_path)
        logger.info(f"Created backup at {backup_path}")
        
        # Write the modified content
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(new_content)
            
        logger.info(f"Successfully marked {file_path} as deprecated")
        return True
        
    except Exception as e:
        logger.error(f"Error marking {file_path} as deprecated: {e}")
        return False

# Example of marking a file as deprecated
# For demonstration only - not actually modifying files
print_section("EXAMPLE DEPRECATION NOTICE")

# Create a sample file to demonstrate deprecation
sample_file = """
# Sample deprecated file
def old_function():
    return "This is the old implementation"

class OldClass:
    def __init__(self):
        self.value = "old"
"""

# Show what the file would look like after adding deprecation notice
alternative = "data_manager.py"
removal_date = "December 31, 2025"

deprecation_notice = f'''"""
DEPRECATED: This file is deprecated and should not be used.
Please use {alternative} instead. It will be removed on {removal_date}.
"""

import warnings
warnings.warn(
    f"The module old_module is deprecated. "
    f"Please use {alternative} instead. It will be removed on {removal_date}.",
    DeprecationWarning,
    stacklevel=2
)

'''

marked_sample = deprecation_notice + sample_file
print("Original file:")
print(sample_file)
print("\nAfter adding deprecation notice:")
print(marked_sample)

# Explain how to use the function in production
print("\nTo mark a real file as deprecated, you would use:")
print("""mark_file_as_deprecated(
    file_path=Path('path/to/file.py'), 
    alternative='new_module.py',
    removal_date='December 31, 2025'
)""")

## 3. Improving Error Handling and Logging

Robust error handling and comprehensive logging are essential for maintainable code, especially in web scraping applications where many things can go wrong (network issues, HTML changes, API changes, etc.).

In this section, we'll demonstrate techniques for:

1. Implementing proper exception handling
2. Creating a consistent logging framework
3. Using decorators for error handling and retry logic
4. Improving error messages and debugging information

In [ ]:
# Set up a comprehensive logging system
def setup_logging(
    log_file: Optional[str] = None,
    console_level: int = logging.INFO,
    file_level: int = logging.DEBUG,
    module_levels: Dict[str, int] = None
) -> logging.Logger:
    """
    Configure a comprehensive logging system with console and file handlers.
    
    Args:
        log_file: Path to log file (optional)
        console_level: Logging level for console output
        file_level: Logging level for file output
        module_levels: Dict of module names and their specific log levels
        
    Returns:
        Configured logger
    """
    print_section("SETTING UP ENHANCED LOGGING")
    
    # Create a custom formatter
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Configure root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.DEBUG)  # Capture all logs
    
    # Remove existing handlers if any
    for handler in root_logger.handlers[:]:
        root_logger.removeHandler(handler)
    
    # Create console handler
    console = logging.StreamHandler()
    console.setLevel(console_level)
    console.setFormatter(formatter)
    root_logger.addHandler(console)
    
    # Create file handler if log_file is provided
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setLevel(file_level)
        file_handler.setFormatter(formatter)
        root_logger.addHandler(file_handler)
        print(f"Logging to file: {log_file}")
    
    # Set specific levels for modules if provided
    if module_levels:
        for module, level in module_levels.items():
            logging.getLogger(module).setLevel(level)
            print(f"Set {module} logging level to {logging.getLevelName(level)}")
    
    logger = logging.getLogger(__name__)
    logger.info("Logging system initialized")
    return logger

# Example usage of the logging setup
log_dir = Path("./logs")
log_dir.mkdir(exist_ok=True)

log_file = log_dir / f"scraper_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure specific levels for different modules
module_levels = {
    'urllib3': logging.WARNING,  # Less verbose output from HTTP library
    'selenium': logging.WARNING, # Less verbose output from Selenium
    'retrievers': logging.DEBUG, # More detailed logs from our retriever module
    'data_manager': logging.DEBUG  # More detailed logs from our data manager
}

# Set up logging
enhanced_logger = setup_logging(
    log_file=str(log_file), 
    console_level=logging.INFO,
    file_level=logging.DEBUG,
    module_levels=module_levels
)

# Example log messages
enhanced_logger.debug("This is a debug message - only appears in log file")
enhanced_logger.info("This is an info message - appears in console and log file")
enhanced_logger.warning("This is a warning message")
enhanced_logger.error("This is an error message")

print(f"Full logs are being saved to: {log_file}")

In [ ]:
# Create error handling decorators
def retry(
    max_tries: int = 3, 
    delay: float = 1.0, 
    backoff: float = 2.0, 
    exceptions: tuple = (Exception,)
):
    """
    Retry decorator with exponential backoff for functions that might fail temporarily.
    
    Args:
        max_tries: Maximum number of attempts
        delay: Initial delay between retries in seconds
        backoff: Backoff multiplier (how much to increase delay each retry)
        exceptions: Tuple of exceptions to catch and retry
        
    Returns:
        Decorated function with retry logic
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            mtries, mdelay = max_tries, delay
            
            # Get function details for better logging
            module = func.__module__
            qualname = func.__qualname__
            full_name = f"{module}.{qualname}"
            
            while mtries > 0:
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    mtries -= 1
                    if mtries <= 0:
                        logger.error(f"Function {full_name} failed after {max_tries} attempts. Error: {e}")
                        raise
                    
                    logger.warning(
                        f"Function {full_name} failed. Retrying in {mdelay:.1f} seconds... "
                        f"({max_tries - mtries}/{max_tries}) Error: {e}"
                    )
                    
                    time.sleep(mdelay)
                    mdelay *= backoff
            return func(*args, **kwargs)
        return wrapper
    return decorator

# Example error handling decorator for web scraping functions
def safe_scraping(func):
    """Decorator to handle common web scraping errors with appropriate logging"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Get function details for better logging
        module = func.__module__
        qualname = func.__qualname__
        full_name = f"{module}.{qualname}"
        
        try:
            return func(*args, **kwargs)
        except (ConnectionError, TimeoutError) as e:
            logger.error(f"Network error in {full_name}: {e}")
            raise
        except Exception as e:
            logger.exception(f"Unexpected error in {full_name}: {e}")
            raise
    return wrapper

# Example of using the decorators
print_section("ERROR HANDLING DECORATOR EXAMPLES")

# Define a mock function to demonstrate the retry decorator
def mock_retrieval_function(success_on_attempt=3):
    """Mock function that simulates a web request that fails initially"""
    mock_retrieval_function.attempts = getattr(mock_retrieval_function, 'attempts', 0) + 1
    
    if mock_retrieval_function.attempts < success_on_attempt:
        raise ConnectionError(f"Connection failed on attempt {mock_retrieval_function.attempts}")
    
    return "Data successfully retrieved"

# Apply our retry decorator
retry_example = retry(max_tries=5, delay=0.1, exceptions=(ConnectionError, TimeoutError))(mock_retrieval_function)

# Reset the counter and try the function
mock_retrieval_function.attempts = 0

try:
    print("Calling function with retry decorator...")
    result = retry_example(success_on_attempt=3)
    print(f"Result: {result}")
except Exception as e:
    print(f"Failed with error: {e}")

print("\nExample implementation of decorated function in production code:")
print("""
@retry(max_tries=3, delay=2.0, exceptions=(ConnectionError, TimeoutError))
@safe_scraping
def retrieve_property_details(self, property_id):
    \"\"\"Retrieve details for a specific property\"\"\"
    url = f"https://example.com/api/properties/{property_id}"
    
    logger.info(f"Retrieving property details for ID: {property_id}")
    response = requests.get(url, timeout=10)
    
    if response.status_code != 200:
        logger.warning(f"Failed to retrieve property {property_id}. Status: {response.status_code}")
        response.raise_for_status()
        
    data = response.json()
    logger.debug(f"Successfully retrieved data for property {property_id}")
    
    return data
""")

In [ ]:
import time

# Create custom exceptions for better error handling
print_section("IMPLEMENTING CUSTOM EXCEPTIONS")

class RealEstateScraperError(Exception):
    """Base exception for all real estate scraper errors."""
    pass

class NetworkError(RealEstateScraperError):
    """Exception raised for network-related errors."""
    pass

class ParseError(RealEstateScraperError):
    """Exception raised when unable to parse data."""
    pass

class APIError(RealEstateScraperError):
    """Exception raised for API-related errors."""
    def __init__(self, status_code, message):
        self.status_code = status_code
        self.message = message
        super().__init__(f"API Error {status_code}: {message}")

class RateLimitError(APIError):
    """Exception raised when rate limited by the API."""
    def __init__(self, retry_after=None):
        self.retry_after = retry_after
        message = f"Rate limit exceeded. Retry after {retry_after} seconds." if retry_after else "Rate limit exceeded."
        super().__init__(429, message)

# Example of using custom exceptions
def demo_api_call(endpoint, fail_type=None):
    """Demonstrate API call with custom exceptions"""
    logger.info(f"Calling API endpoint: {endpoint}")
    
    # Simulate different failure scenarios
    if fail_type == 'network':
        logger.error("Network error: Connection refused")
        raise NetworkError("Connection refused")
    elif fail_type == 'parse':
        logger.error("Parse error: Invalid JSON response")
        raise ParseError("Invalid JSON response")
    elif fail_type == 'api_400':
        logger.error("API error: Bad request")
        raise APIError(400, "Bad request")
    elif fail_type == 'rate_limit':
        retry_after = 30
        logger.error(f"Rate limit error: Retry after {retry_after} seconds")
        raise RateLimitError(retry_after)
    
    return {"status": "success", "data": {"endpoint": endpoint}}

# Example function with comprehensive error handling
def get_property_with_error_handling(property_id):
    """Get property details with comprehensive error handling"""
    endpoint = f"/properties/{property_id}"
    
    try:
        result = demo_api_call(endpoint)
        logger.info(f"Successfully retrieved property {property_id}")
        return result['data']
        
    except NetworkError as e:
        logger.error(f"Network error while retrieving property {property_id}: {e}")
        return {"error": "network", "message": str(e)}
        
    except ParseError as e:
        logger.error(f"Parse error while retrieving property {property_id}: {e}")
        return {"error": "parse", "message": str(e)}
        
    except RateLimitError as e:
        logger.warning(f"Rate limited while retrieving property {property_id}: {e}")
        if e.retry_after:
            logger.info(f"Waiting {e.retry_after} seconds before retrying...")
            time.sleep(e.retry_after)
            # In a real implementation, we might retry here
        return {"error": "rate_limit", "retry_after": e.retry_after}
        
    except APIError as e:
        logger.error(f"API error while retrieving property {property_id}: {e}")
        return {"error": "api", "status_code": e.status_code, "message": e.message}
        
    except Exception as e:
        logger.exception(f"Unexpected error while retrieving property {property_id}: {e}")
        return {"error": "unknown", "message": str(e)}

# Demonstrate the error handling with different scenarios
print("Successful API call:")
print(get_property_with_error_handling("12345"))

print("\nNetwork error scenario:")
print(get_property_with_error_handling("12345", fail_type="network"))

print("\nRate limit error scenario:")
print(get_property_with_error_handling("12345", fail_type="rate_limit"))

print("\nImplementation example for custom error handling:")
print("""
def retrieve_property_data(self, property_id):
    try:
        url = f"{self.base_url}/properties/{property_id}"
        response = self.session.get(url, timeout=self.timeout)
        
        if response.status_code == 429:
            retry_after = int(response.headers.get('Retry-After', 60))
            raise RateLimitError(retry_after)
            
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data:
            raise APIError(response.status_code, data['error'])
            
        return self._parse_property_data(data)
        
    except json.JSONDecodeError as e:
        raise ParseError(f"Invalid JSON response: {e}")
        
    except requests.ConnectionError as e:
        raise NetworkError(f"Connection error: {e}")
        
    except requests.Timeout as e:
        raise NetworkError(f"Request timed out: {e}")
""")

## 4. Updating Documentation

Keeping documentation up-to-date is crucial for maintaining a healthy codebase. As we refactor code and mark files as deprecated, we need to ensure that our documentation reflects these changes.

In this section, we'll demonstrate:

1. Scanning documentation for references to deprecated files
2. Updating docstrings to follow best practices
3. Generating up-to-date documentation with Sphinx
4. Creating example notebooks like this one

In [ ]:
# Function to scan documentation for references to deprecated files
def scan_docs_for_references(
    docs_dir: Path, 
    deprecated_files: List[str],
    file_patterns: List[str] = ['*.md', '*.rst', '*.ipynb']
) -> Dict[str, List[Dict]]:
    """
    Scan documentation files for references to deprecated modules or files.
    
    Args:
        docs_dir: Directory containing documentation
        deprecated_files: List of deprecated file/module names
        file_patterns: List of file patterns to search
        
    Returns:
        Dictionary mapping documentation files to references found
    """
    print_section("SCANNING DOCUMENTATION FOR DEPRECATED REFERENCES")
    
    if not docs_dir.exists() or not docs_dir.is_dir():
        logger.error(f"Directory {docs_dir} does not exist")
        return {}
        
    # Get all documentation files
    doc_files = []
    for pattern in file_patterns:
        doc_files.extend(list(docs_dir.glob(f"**/{pattern}")))
        
    logger.info(f"Found {len(doc_files)} documentation files in {docs_dir}")
    
    # Prepare results
    results = {}
    
    # Look for references to deprecated files
    for doc_file in doc_files:
        references = []
        
        # Different handling based on file type
        if doc_file.suffix == '.ipynb':
            # For Jupyter notebooks, we need to parse JSON
            try:
                import json
                with open(doc_file, 'r', encoding='utf-8') as f:
                    notebook = json.load(f)
                    
                # Check all cells
                for i, cell in enumerate(notebook.get('cells', [])):
                    if cell.get('cell_type') in ['code', 'markdown']:
                        content = ''.join(cell.get('source', []))
                        
                        for dep_file in deprecated_files:
                            if dep_file in content:
                                references.append({
                                    'file': dep_file,
                                    'cell': i + 1,
                                    'excerpt': content[:100] + '...' if len(content) > 100 else content
                                })
            except Exception as e:
                logger.error(f"Error processing notebook {doc_file}: {e}")
                
        else:
            # For markdown and RST files
            try:
                with open(doc_file, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    
                for dep_file in deprecated_files:
                    # Find all occurrences
                    for match in re.finditer(re.escape(dep_file), content):
                        # Get context around the reference
                        start = max(0, match.start() - 50)
                        end = min(len(content), match.end() + 50)
                        excerpt = content[start:end]
                        
                        references.append({
                            'file': dep_file,
                            'position': match.start(),
                            'excerpt': '...' + excerpt + '...' if start > 0 or end < len(content) else excerpt
                        })
            except Exception as e:
                logger.error(f"Error processing file {doc_file}: {e}")
                
        if references:
            results[str(doc_file.relative_to(docs_dir))] = references
            
    return results

# Example usage - scan documentation for references to deprecated files
# Define the documentation directory
docs_dir = Path("../../docs")

# Define deprecated files/modules
deprecated_files = [
    "fetch_ads.py", 
    "models.py", 
    "main.py"
]

# Scan for references if docs directory exists
doc_references = None
if docs_dir.exists():
    doc_references = scan_docs_for_references(docs_dir, deprecated_files)
    
    if doc_references:
        print(f"Found references to deprecated files in {len(doc_references)} documentation files:")
        for doc_file, references in doc_references.items():
            print(f"\n{doc_file} - {len(references)} references:")
            for ref in references:
                print(f"  - Reference to {ref['file']}:")
                print(f"    {ref['excerpt'].strip()}")
                print()
    else:
        print("No references to deprecated files found in documentation.")
else:
    print(f"Documentation directory {docs_dir} not found. Skipping scan.")

# Tips for updating documentation
print_section("DOCUMENTATION BEST PRACTICES")

print("""When updating documentation, follow these best practices:

1. Remove references to deprecated modules/files
2. Update code examples to use the current API
3. Use proper docstrings for all functions and classes
4. Include example usage in docstrings
5. Document error handling and exceptions
6. Create example notebooks for common use cases
7. Update the main README.md file to reflect changes
8. Regenerate API documentation using Sphinx

Example docstring format (Google style):
```python
def get_property_details(property_id: str, include_images: bool = False) -> Dict[str, Any]:
    \"\"\"
    Retrieve detailed information about a specific property.
    
    Args:
        property_id: The unique identifier of the property
        include_images: Whether to include image URLs in the response
        
    Returns:
        Dictionary containing property details
        
    Raises:
        NetworkError: If there's a connection issue
        APIError: If the API returns an error response
        
    Example:
        >>> details = get_property_details("12345")
        >>> print(details["price"])
        250000
    \"\"\"
```

Remember to regenerate Sphinx documentation after updating docstrings:

```bash
cd ../../docs
make html
```
""")

## 5. Conclusion and Recommended Workflow

In this notebook, we've covered several essential maintenance tasks for our real estate scraping project:

1. **Identifying and Handling Deprecated Files**:
   - Scanning the codebase for potentially deprecated files
   - Adding proper deprecation notices with alternatives
   - Managing the transition from old to new code

2. **Improving Error Handling and Logging**:
   - Setting up comprehensive logging
   - Creating retry decorators for network operations
   - Implementing custom exceptions for better error handling

3. **Updating Documentation**:
   - Scanning for references to deprecated files
   - Following documentation best practices
   - Keeping examples and notebooks up-to-date

### Recommended Maintenance Workflow

When maintaining the codebase, follow this workflow to ensure smooth transitions and minimal disruption:

1. **Identify Changes Needed**: Scan the codebase for deprecated or problematic files
2. **Plan the Transition**: Determine which files need updating and which need replacing
3. **Implement New Functionality**: Create new modules with improved implementations
4. **Mark Old Files as Deprecated**: Add deprecation notices with clear alternatives
5. **Update Documentation**: Ensure all documentation reflects the current state
6. **Update Tests**: Make sure tests use the new modules instead of deprecated ones
7. **Monitor Usage**: Track how often deprecated modules are still being used
8. **Remove Deprecated Files**: After a suitable transition period, remove deprecated files

Regular maintenance using these techniques will keep the codebase healthy, maintainable, and up-to-date with best practices.